In [ ]:
!python -m pip install requests

In [ ]:

import json
from datetime import datetime, timezone
from pathlib import Path

import requests

OUTPUT_DIR = Path("outputs/sib_pilot")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIMEOUT = 30

# API documentada del Catálogo de la Biodiversidad.
CATALOG_URL = "https://api.catalogo.biodiversidad.co/record_search/search"

# Endpoint de registros biológicos divulgado por SiB Colombia.
# Su disponibilidad y parámetros actuales se deben comprobar.
OCCURRENCE_URL = "https://api.biodiversidad.co/occurrence/search"


def probe_api(name, url, params):
    print(f"\nConsultando: {name}")
    result = {
        "service": name,
        "url": url,
        "params": params,
        "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    try:
        response = requests.get(
            url,
            params=params,
            headers={"Accept": "application/json, application/geo+json"},
            timeout=TIMEOUT,
        )

        result["http_status"] = response.status_code
        result["final_url"] = response.url
        result["content_type"] = response.headers.get("Content-Type")

        response.raise_for_status()

        try:
            payload = response.json()
            result["response_type"] = type(payload).__name__

            # Guardar la respuesta íntegra para inspección.
            output_file = OUTPUT_DIR / f"{name}_raw.json"
            output_file.write_text(
                json.dumps(payload, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )

            result["raw_file"] = str(output_file)
            if isinstance(payload, dict):
                result["top_level_keys"] = list(payload.keys())
                result["features_count"] = len(payload.get("features", []))
                result["results_count"] = len(payload.get("results", []))
            elif isinstance(payload, list):
                result["items_count"] = len(payload)

            result["status"] = "success"
            print(f"Respuesta HTTP: {response.status_code}")
            print(f"Respuesta guardada en: {output_file}")

        except ValueError:
            result["status"] = "response_not_json"
            result["response_preview"] = response.text[:1000]
            print("La respuesta no es JSON; se guardará una muestra.")

    except requests.RequestException as error:
        result["status"] = "error"
        result["error"] = str(error)

        if getattr(error, "response", None) is not None:
            result["response_preview"] = error.response.text[:1000]

        print(f"Consulta fallida: {error}")

    return result


def main():
    results = []

    # Prueba de información descriptiva de especies.
    results.append(
        probe_api(
            "sib_species",
            CATALOG_URL,
            {"q": "Leopardus pardalis", "size": 5},
        )
    )

    # Prueba exploratoria del servicio de ocurrencias.
    # No asumimos que los parámetros antiguos sigan vigentes.
    results.append(
        probe_api(
            "sib_occurrences",
            OCCURRENCE_URL,
            {"country": "CO", "limit": 10},
        )
    )

    report = {
        "run_at_utc": datetime.now(timezone.utc).isoformat(),
        "purpose": "Diagnóstico inicial de servicios SiB Colombia",
        "results": results,
        "note": (
            "Una consulta exitosa confirma conectividad, no la "
            "completitud ni la validez científica de los datos."
        ),
    }

    report_path = OUTPUT_DIR / "diagnostic_summary.json"
    report_path.write_text(
        json.dumps(report, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print("\n--- RESUMEN ---")
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print(f"\nInforme: {report_path.resolve()}")


if __name__ == "__main__":
    main()



Consultando: sib_species
Respuesta HTTP: 200
Respuesta guardada en: outputs/sib_pilot/sib_species_raw.json

Consultando: sib_occurrences
Consulta fallida: HTTPSConnectionPool(host='api.biodiversidad.co', port=443): Max retries exceeded with url: /occurrence/search?country=CO&limit=10 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x795bb0e2dbd0>: Failed to resolve 'api.biodiversidad.co' ([Errno -2] Name or service not known)"))

--- RESUMEN ---
{
  "run_at_utc": "2026-10-21T01:35:51.007256+00:00",
  "purpose": "Diagnóstico inicial de servicios SiB Colombia",
  "results": [
    {
      "service": "sib_species",
      "url": "https://api.catalogo.biodiversidad.co/record_search/search",
      "params": {
        "q": "Leopardus pardalis",
        "size": 5
      },
      "checked_at_utc": "2026-10-21T01:35:49.452991+00:00",
      "http_status": 200,
      "final_url": "https://api.catalogo.biodiversidad.co/record_search/search?q=Leopardus+pardalis&size=5",
